# 🎵 SPOTIFY AUDIO TRENDS ANALYTICS — PREPROCESSING PIPELINE
## Đồ án Cuối kỳ: Môn Tương tác Dữ liệu (Interactive Data Visualization)

---

### 📌 THÔNG TIN HỌC THUẬT & PHÂN CÔNG NHIỆM VỤ
- **Thành viên thực hiện:** Đinh Quốc Anh (MSSV: 24133003) — *Vai trò: Data Engineer & Data Analyst*
- **Trọng tâm phụ trách:** Tiền xử lý đa nguồn, Kết nối 5 bảng dữ liệu (JOIN), Xây dựng 13 Calculated Fields & Kiểm định chất lượng dữ liệu
- **Chuẩn Barem Rubric:** Đáp ứng trọn vẹn **Mục 1: Thu thập & Tiền xử lý dữ liệu (2.5 / 2.5 điểm)**:
  - *Xác định bài toán & Dataset (0.5đ):* 5 tập dữ liệu (>1.9M bản ghi thô), Data Dictionary rõ ràng.
  - *Làm sạch dữ liệu (0.5đ):* Xử lý missing, loại ngoại lai, chuẩn hóa chuỗi, regex tách số doanh số.
  - *Biến đổi dữ liệu (0.75đ):* Kiến trúc Master Dataset hợp nhất 41 cột phục vụ Insight, 4 phép JOIN có thứ tự, tạo 13 Calculated Fields.
  - *Khám phá dữ liệu EDA tĩnh (0.75đ):* Biểu đồ Seaborn/Matplotlib kiểm định phân phối và ma trận tương quan.

---

### 🏗️ KIẾN TRÚC MASTER DATASET HỢP NHẤT & CÁC TỆP SAMPLE
Pipeline hợp nhất toàn bộ dữ liệu vào **1 Tệp Master lớn duy nhất** và sinh các tệp sample gọn nhẹ cho nhóm:
1. **File Master duy nhất (`merged_spotify_data.csv`):** 1,078,125 dòng × 41 cột (~407.4 MB), tích hợp cả dữ liệu Kinh tế - Xã hội 74 nước và Dòng thời gian Đại dịch COVID-19 (`covid_phase`), lưu giữ chính xác 41 cột cốt lõi phục vụ Insight.
2. **File Sample Dashboard (`spotify_sample_merged.csv`):** 20,000 dòng × 41 cột (~7.6 MB), phục vụ dựng UI Dashboard và kiểm thử biểu đồ Plotly trong 0.2 giây.
3. **File Sample ML cho Thành Tài (`tracks_sample_ml.csv`):** 36,699 bài hát độc nhất × 18 cột (~5.7 MB), chuẩn hóa cấp độ bài hát (Song-level) chống 100% Data Leakage cho mô hình Logistic Regression.

In [ ]:
# ════════════════════════════════════════════════════════════════
# 1. THIẾT LẬP MÔI TRƯỜNG & IMPORT THƯ VIỆN
# ════════════════════════════════════════════════════════════════
import os
import sys
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")

# Cấu hình đường dẫn root dự án
NOTEBOOK_DIR = os.path.dirname(os.path.abspath("__file__")) if "__file__" in locals() else os.getcwd()
PROJECT_ROOT = os.path.abspath(os.path.join(NOTEBOOK_DIR, "..")) if os.path.basename(NOTEBOOK_DIR) == "notebooks" else NOTEBOOK_DIR

if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

# Import các hàm từ module data_loader
from src.data_loader import (
    load_raw_data,
    clean_raw_charts,
    clean_raw_tracks,
    clean_raw_artists,
    clean_raw_gdp,
    clean_raw_country_codes,
    build_master_dataset,
    generate_sample_datasets,
    load_master_data,
    load_sample_data,
    load_ml_sample,
    data_quality_report,
    RAW_DIR, PROCESSED_DIR, SAMPLE_DIR
)

# Cấu hình giao diện biểu đồ Seaborn
sns.set_theme(style="whitegrid", font="sans-serif")
plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["font.size"] = 11

print(f"✅ Thư mục dự án: {PROJECT_ROOT}")
print(f"✅ Phiên bản Pandas: {pd.__version__} | NumPy: {np.__version__}")

---
## 📂 BƯỚC 1: NẠP VÀ ĐÁNH GIÁ CHẤT LƯỢNG 5 BẢNG DỮ LIỆU THÔ
Theo yêu cầu Rubric, đồ án sử dụng **5 tập dữ liệu đa nguồn** với tổng quy mô **hơn 1.9 triệu bản ghi**:
1. `spotify-weekly-charts.csv`: Bảng xếp hạng tuần toàn cầu và 74 quốc gia (~1.78M dòng × 36 cột).
2. `spotify-tracks-dataset.csv`: 114,000 bài hát với 9 đặc trưng âm học và thể loại.
3. `gdp_data.csv`: Chuỗi thời gian GDP các quốc gia từ World Bank (1960–2022).
4. `country_codes.csv`: Phân loại khu vực và nhóm thu nhập World Bank của 217 quốc gia.
5. `best-selling-artists.csv`: 121 nghệ sĩ bán đĩa chạy nhất lịch sử âm nhạc.

In [ ]:
# Nạp 5 tập dữ liệu thô
charts_raw, tracks_raw, gdp_raw, country_codes_raw, artists_raw = load_raw_data()

In [ ]:
# Xuất bảng báo cáo chất lượng dữ liệu (Data Quality Profile) cho 5 bảng thô
print("=== BÁO CÁO CHẤT LƯỢNG DỮ LIỆU THÔ ===")
raw_tables = [
    ("Weekly Charts", charts_raw),
    ("Tracks Dataset", tracks_raw),
    ("GDP Data", gdp_raw),
    ("Country Codes", country_codes_raw),
    ("Best-Selling Artists", artists_raw)
]
for name, df_obj in raw_tables:
    dup_count = df_obj.duplicated().sum()
    null_pct = df_obj.isnull().mean().mean() * 100
    print(f"• {name:<22s}: {df_obj.shape[0]:>10,} dòng × {df_obj.shape[1]:>2d} cột | Null trung bình: {null_pct:>5.1f}% | Trùng lặp: {dup_count:,} dòng")

---
## 🧹 BƯỚC 2: TIỀN XỬ LÝ & LÀM SẠCH ĐỘC LẬP TỪNG BẢNG
Các thuật toán làm sạch cụ thể:
* **Weekly Charts:** Trích `track_id` chuẩn từ URI; chuyển `week` sang Datetime; ép kiểu số cho `streams`, loại `streams <= 0`; ép kiểu numeric cho 9 audio features; khử trùng lặp `(track_id, week, country)` và loại bỏ sớm 15 cột rác.
* **Tracks Dataset:** Dedup `track_id` theo `popularity` cao nhất (từ 114k -> 89.7k bài); kiểm định biên độ âm học trong $[0, 1]$; clip `loudness` $[-60, 5]$ dB, `tempo` $[0, 300]$ BPM.
* **Artists Dataset:** Regex bóc tách doanh số đĩa triệu bản; chuẩn hóa tên nghệ sĩ lower-case.
* **GDP & Country Codes:** Chuẩn hóa tên trường (`gdp_value`, `region_wb`, `income_group`).

In [ ]:
# Thực hiện làm sạch độc lập từng bảng
charts_clean = clean_raw_charts(charts_raw)
tracks_clean = clean_raw_tracks(tracks_raw)
artists_clean = clean_raw_artists(artists_raw)
gdp_clean = clean_raw_gdp(gdp_raw)
country_codes_clean = clean_raw_country_codes(country_codes_raw)

---
## 🌍 BƯỚC 3: XÂY DỰNG MASTER DATASET HỢP NHẤT (41 CỘT INSIGHT)
Quy trình kết nối dữ liệu:
1. **Lọc chuỗi thời gian trọng tâm (2020 – 2022):** Bao quát cả 3 pha đại dịch COVID-19 và số liệu bảng xếp hạng 74 quốc gia.
2. **4 phép LEFT JOIN có thứ tự:**
   * *Charts ⋈ GDP:* trên `(country, year) == (country_name, year)`
   * *+ Country Codes:* trên `country_code`
   * *+ Tracks Dataset:* trên `track_id` (lấy `track_genre`, `popularity`)
   * *+ Best-Selling Artists:* trên tên nghệ sĩ chuẩn hóa (đánh dấu `is_legacy_artist`)
3. **Xây dựng 13 Calculated Fields:** `covid_phase`, `mood_category`, `stream_share_country`, `energy_loudness_index`, `is_latin`, `is_kpop`, `is_trending`, `is_popular`, `gdp_log`, `log_streams`, `audio_diversity_score`, `duration_min`.
4. **Lọc giữ chính xác 41 cột cốt lõi phục vụ Insight**.

In [ ]:
# Xây dựng Master Dataset hợp nhất (chỉ giữ 41 cột phục vụ Insight)
df_master = build_master_dataset(
    charts_clean, gdp_clean, country_codes_clean, tracks_clean, artists_clean, save=False
)
print(f"\n📊 Master Dataset Shape: {df_master.shape[0]:,} dòng × {df_master.shape[1]} cột")
print(f"🔹 Tỷ lệ khớp GDP (74 nước): {df_master[df_master['country'] != 'Global']['gdp_value'].notna().mean()*100:.1f}%")
print(f"🔹 Số quốc gia có mặt: {df_master['country'].nunique():,} (bao gồm 74 nước và Global)")

---
## 🔬 BƯỚC 4: SINH CÁC TẬP SAMPLE CHO PHÁT TRIỂN & MACHINE LEARNING
Để thuận tiện cho làm việc nhóm mà không bị quá tải RAM:
1. **`spotify_sample_merged.csv`:** 20,000 dòng có đầy đủ 41 cột, phục vụ dựng giao diện Streamlit Dashboard và kiểm thử nhanh biểu đồ Plotly.
2. **`tracks_sample_ml.csv`:** Dữ liệu cấp độ bài hát (Song-level) độc nhất gom nhóm theo `track_id`, chống 100% Data Leakage cho Đỗ Thanh Thành Tài huấn luyện mô hình Logistic Regression.

In [ ]:
# Sinh 2 tệp sample
df_sample, df_ml = generate_sample_datasets(df_master, sample_size=20000, save=False)
print(f"✅ Sample Dashboard: {df_sample.shape[0]:,} dòng × {df_sample.shape[1]} cột")
print(f"✅ ML Dataset (Thành Tài): {df_ml.shape[0]:,} bài hát độc nhất × {df_ml.shape[1]} cột | Tỷ lệ Hit: {df_ml['is_trending'].mean()*100:.1f}%")

---
## 🧮 BƯỚC 5: KIỂM ĐỊNH CHẤT LƯỢNG CÁC TRƯỜNG TÍNH TOÁN (CALCULATED FIELDS)
Đánh giá tính hợp lý và sự phân bổ của các trường dữ liệu sinh mới phục vụ Insight đồ án.

In [ ]:
# 1. Phân bổ các giai đoạn đại dịch COVID-19
print("🔹 PHÂN BỔ 3 GIAI ĐOẠN ĐẠI DỊCH (COVID PHASE):")
phase_dist = df_master["covid_phase"].value_counts(dropna=False)
for p, count in phase_dist.items():
    print(f"  • {p:<25s}: {count:>10,} bản ghi ({count/len(df_master)*100:>5.1f}%)")

# 2. Phân bổ 4 góc phần tư tâm trạng Russell
print("\n🔹 PHÂN BỔ BẢN ĐỒ CẢM XÚC RUSSELL (MOOD CATEGORY):")
mood_dist = df_master["mood_category"].value_counts(dropna=False)
for m, count in mood_dist.items():
    print(f"  • {m:<25s}: {count:>10,} bản ghi ({count/len(df_master)*100:>5.1f}%)")

# 3. Thống kê bài hát thuộc làn sóng âm nhạc toàn cầu
print("\n🔹 TỶ LỆ LÀN SÓNG ÂM NHẠC TOÀN CẦU:")
print(f"  • Nhạc Latin: {df_master['is_latin'].sum():,} bản ghi ({df_master['is_latin'].mean()*100:.1f}%)")
print(f"  • K-Pop:      {df_master['is_kpop'].sum():,} bản ghi ({df_master['is_kpop'].mean()*100:.1f}%)")
print(f"  • Nghệ sĩ huyền thoại (Legacy): {df_master['is_legacy_artist'].sum():,} bản ghi")

---
## 📊 BƯỚC 6: KHÁM PHÁ DỮ LIỆU TĨNH BAN ĐẦU (EDA SANITY CHECKS)
Thực hiện 3 biểu đồ Seaborn tĩnh đáp ứng tiêu chí **Rubric 1.4: Khám phá dữ liệu EDA tĩnh (0.75 điểm)**.

In [ ]:
# Biểu đồ 1: Ma trận nhiệt tương quan Pearson (Heatmap) giữa 9 Audio Features & Lượt nghe
audio_cols = ["danceability", "energy", "loudness", "speechiness", "acousticness", "instrumentalness", "liveness", "valence", "tempo", "log_streams"]
corr_matrix = df_sample[audio_cols].corr()

plt.figure(figsize=(10, 8))
sns.heatmap(corr_matrix, annot=True, fmt=".2f", cmap="coolwarm", cbar=True, square=True, linewidths=0.5)
plt.title("Biểu đồ 1: Ma trận Tương quan Pearson giữa các Đặc trưng Âm thanh & Lượt nghe (Log Streams)", fontsize=13, fontweight="bold", pad=15)
plt.tight_layout()
plt.show()

In [ ]:
# Biểu đồ 2: So sánh đặc trưng âm thanh qua 3 giai đoạn COVID-19 (Bảng Global)
covid_global = df_master[df_master["country"] == "Global"]
phase_stats = covid_global.groupby("covid_phase")[["energy", "valence", "danceability", "acousticness"]].mean().reset_index()
phase_order = ["Lockdown & Shock", "Vaccine & Escapism", "Reopening & New Normal"]
phase_stats = phase_stats.set_index("covid_phase").reindex([p for p in phase_order if p in phase_stats["covid_phase"].values]).reset_index()

df_melt = phase_stats.melt(id_vars="covid_phase", var_name="Feature", value_name="Mean_Score")
plt.figure(figsize=(11, 5))
sns.barplot(data=df_melt, x="covid_phase", y="Mean_Score", hue="Feature", palette="tab10")
plt.title("Biểu đồ 2: Biến động Giá trị Trung bình các Đặc trưng Âm thanh qua 3 Pha Đại dịch COVID-19 (Global)", fontsize=12, fontweight="bold")
plt.xlabel("Giai đoạn Đại dịch", fontweight="bold")
plt.ylabel("Điểm số trung bình [0.0 - 1.0]", fontweight="bold")
plt.ylim(0, 0.85)
plt.legend(title="Audio Feature", loc="upper right")
plt.tight_layout()
plt.show()

In [ ]:
# Biểu đồ 3: Phân phối Danceability và Energy theo Nhóm thu nhập World Bank
df_income = df_sample.dropna(subset=["income_group"])
income_order = ["Low income", "Lower middle income", "Upper middle income", "High income"]
order_filtered = [inc for inc in income_order if inc in df_income["income_group"].unique()]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.boxplot(data=df_income, x="income_group", y="danceability", order=order_filtered, ax=axes[0], palette="Blues")
axes[0].set_title("Độ dễ nhảy (Danceability) theo Nhóm thu nhập", fontweight="bold")
axes[0].set_xlabel("Nhóm thu nhập (World Bank)")
axes[0].tick_params(axis="x", rotation=15)

sns.boxplot(data=df_income, x="income_group", y="energy", order=order_filtered, ax=axes[1], palette="Reds")
axes[1].set_title("Năng lượng âm nhạc (Energy) theo Nhóm thu nhập", fontweight="bold")
axes[1].set_xlabel("Nhóm thu nhập (World Bank)")
axes[1].tick_params(axis="x", rotation=15)

plt.suptitle("Biểu đồ 3: So sánh Phân phối Âm thanh giữa các Phân tầng Thu nhập Quốc gia", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

---
## 📦 BƯỚC 7: TỔNG KẾT & QUẢN TRỊ TỆP DỮ LIỆU ĐẦU RA
Toàn bộ quá trình tiền xử lý hoàn tất, hệ thống đã sẵn sàng cung cấp dữ liệu cho Dashboard và mô hình Machine Learning.

In [ ]:
# Bảng tổng hợp các tệp dữ liệu đầu ra
summary_files = [
    ("Master Dataset (Toàn bộ dữ liệu Insight)", os.path.join(PROJECT_ROOT, "data", "processed", "merged_spotify_data.csv")),
    ("Sample Dashboard (41 cột cho Streamlit)", os.path.join(PROJECT_ROOT, "data", "sample", "spotify_sample_merged.csv")),
    ("Sample ML (Song-level cho Thành Tài)", os.path.join(PROJECT_ROOT, "data", "sample", "tracks_sample_ml.csv"))
]

print("=" * 80)
print(f"{'Tên tệp dữ liệu':<40s} | {'Số dòng':<12s} | {'Số cột':<8s} | {'Dung lượng':<12s}")
print("=" * 80)
for name, fpath in summary_files:
    if os.path.exists(fpath):
        fmb = os.path.getsize(fpath) / (1024 * 1024)
        # Đọc nhanh header & length
        tmp = pd.read_csv(fpath, nrows=2)
        with open(fpath, "rb") as f:
            lines = sum(1 for _ in f) - 1
        print(f"{name:<40s} | {lines:>12,} | {len(tmp.columns):>8d} | {fmb:>9.1f} MB")
    else:
        print(f"{name:<40s} | Chưa sinh file")
print("=" * 80)